In [86]:
#Gompertz model

#Model
def Gompertz(t, k1, k2, k3, k4, k5, k6):
    N_0 = N[0]
    return np.log(N_0) + k1 * np.exp(-np.exp(-k2 * (t - k3))) - k4 * np.exp(-np.exp(-k5 * (t - k6)))
    
# Estimate initial values for Gompertz
def estimate_initial_gompertz(t, N):
    N_0 = N[0]
    N_max = np.max(N)

    k1_init = np.log(N_max / N_0)
    k2_init = 1 / (t[np.argmax(np.diff(N))] - t[0])
    k3_init = t[np.argmax(np.diff(N))] / 2
    k4_init = k1_init / 2
    k5_init = k2_init
    k6_init = k3_init
    return [k1_init, k2_init, k3_init, k4_init, k5_init, k6_init]
    

In [ ]:
#Churchill

#Model
def Churchill(t, k1, k2, k3, k4, k5, k6):
    return ((1 / k1) * np.exp(k3 * t) + (1 / k2) * np.exp(k4 * t)) ** (-1)
    
# Estimate initial values for Churchill
def estimate_initial_churchill(t, N):
    N_0 = N[0]
    N_max = np.max(N)
    
    if N_max == N_0:
        k1_init = 1.0
        k2_init = 1.0
    else:
        k1_init = 1 / (N_max - N_0)
        k2_init = 1 / (N_max - N_0)
    
    growth_phase = t[:len(t)//2]
    decay_phase = t[len(t)//2:]
    
    if np.ptp(growth_phase) == 0 or np.ptp(decay_phase) == 0:
        k3_init = 1.0
        k4_init = 1.0
    else:
        k3_init = np.log(N_max / N_0) / np.ptp(growth_phase)
        k4_init = np.log(N_max / N_0) / np.ptp(decay_phase)
    
    return [k1_init, k2_init, k3_init, k4_init, 0, 0]


In [ ]:
#Logistic

#Model
def Logistic(t, k1, k2, k3, k4, k5, k6):
    return k1 / (1 + np.exp(-k2 * (t - k3)))

# Estimate initial values for Logistic
def estimate_initial_logistic(t, N):
    N_max = np.max(N)
    N_min = np.min(N)
    
    # Initial guess for k1
    k1_init = N_max - N_min
    
    # Initial guess for k2
    k2_init = 1.0
    
    # Initial guess for k3
    k3_init = np.median(t)
        
    return [k1_init, k2_init, k3_init, 0,0,0]

In [87]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from ipywidgets import interact, FloatSlider, Output, VBox
import ipywidgets as widgets
from IPython.display import clear_output, display




    
# Plotting function
def plot_fit(k1, k2, k3, k4, k5, k6, func):
    plt.scatter(t, N, label='Data')
    plt.plot(t, np.exp(func(t, k1, k2, k3, k4, k5, k6)), 'r-', label='Fit')
    plt.xlim(0, 160)
    plt.ylim(0, 8)
    plt.xticks(np.arange(0, 180, step=20))
    plt.xlabel('t (days)')
    plt.ylabel('log(CFU/ml)')
    plt.legend()
    plt.show()





    
# Processes file and launches interactables
def process_file(file_name, func):
    clear_output(wait=True)
    global t, N
    data = pd.read_csv(file_name, header=None)
    t = data[0]
    N = data[1]
    
    # Estimate initial values
    if func == Gompertz:
        p0 = estimate_initial_gompertz(t, N)
    elif func == Churchill:
        p0 = estimate_initial_churchill(t, N)
    elif func == Logistic:
        p0 = [np.max(N), 1.0, np.median(t),0,0,0]
    else:
        p0 = [1, 1, 1, 1, 1, 1] 
    
    print("Initial parameter estimates:", p0)

    # Fit the model
    try:
        if func == Logistic:
            popt, pcov = curve_fit(func, t, N, p0=p0, maxfev=9999)
        else:
            popt, pcov = curve_fit(func, t, np.log(N), p0=p0, bounds=(0, np.inf), maxfev=9999)
        print("Fitted parameters:", popt)
    except RuntimeError as e:
        print(f"An error occurred during fitting: {e}")
        return
    except ValueError as e:
        print(f"Value error during fitting: {e}")
        return

    # Create sliders for each parameter
    sliders = [FloatSlider(min=0.01, max=100, step=0.001, value=param, description=f'k{i+1}', continuous_update=False) for i, param in enumerate(popt)]

    # Create an output widget
    out = Output()

    # Update plot function
    def update_plot(**kwargs):
        with out:
            clear_output(wait=True)
            plot_fit(kwargs['k1'], kwargs['k2'], kwargs['k3'], kwargs['k4'], kwargs['k5'], kwargs['k6'], func)

    # Create interactive widgets
    interact(update_plot, k1=sliders[0], k2=sliders[1], k3=sliders[2], k4=sliders[3], k5=sliders[4], k6=sliders[5])

    # Display the layout
    box = VBox([out], layout=widgets.Layout(height='500px'))
    display(box)

# Save plot image to .png
def save_file(filename, func):
    k1 = k1_slider.value
    k2 = k2_slider.value
    k3 = k3_slider.value
    k4 = k4_slider.value
    k5 = k5_slider.value
    k6 = k6_slider.value
    plt.scatter(t, N, label='Data')
    plt.plot(t, np.exp(func(t, k1, k2, k3, k4, k5, k6)), 'r-', label='Fit')
    plt.xlabel('t (days)')
    plt.ylabel('log(CFU/ml)')
    plt.legend()
    plt.savefig(filename)  # Save the plot with the given filename

# Example usage
# process_file('your_data_file.csv', Gompertz)


In [88]:
process_file("control.csv", Gompertz)

Initial parameter estimates: [1.2568170281019202, 0.3333333333333333, 1.5, 0.6284085140509601, 0.3333333333333333, 1.5]
Fitted parameters: [ 1.17643862  2.24115774  6.25559615  0.2449459   0.07034968 68.26995583]


interactive(children=(FloatSlider(value=1.1764386217062868, continuous_update=False, description='k1', min=0.0…

In [89]:
process_file('B281.csv', Logistic)


Initial parameter estimates: [7.975, 1.0, 77.5, 0, 0, 0]
Value error during fitting: Inconsistent shapes between bounds and `x0`.
